In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import mygene
import time
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
import gc


In [ ]:
VARIANTS_CSV   = "./clinvar_updated.csv"
VESM_PARQUET   = "./UniProtKB_VESM++_scores.parquet.gzip"
UNIPROT_CACHE  = "nm_to_uniprot_cache.csv"
OUTPUT_CSV     = "benchmark_scores_vesm.csv"

data = pd.read_csv(VARIANTS_CSV, dtype={"#CHROM": str})
print(f"Loaded {len(data)} variants")

In [ ]:
# import pandas as pd

# df = pd.read_parquet("./UniProtKB_VESM++_scores.parquet.gzip")

# print("Columns:", df.columns.tolist())
# print("Total rows:", len(df))

# # 先看欄位名稱再決定用哪個
# print(df.head(3))

In [ ]:
# print("Total rows:", len(df))
# print("Unique uniprot_id:", df["uniprot_id"].nunique())
# print("Unique (uniprot_id, protein_variant) pairs:", df.drop_duplicates(subset=["uniprot_id", "protein_variant"]).shape[0])
# print("Any duplicates:", df.duplicated(subset=["uniprot_id", "protein_variant"]).any())

In [ ]:
# ══════════════════════════════════════════════════════════════
# RefSeq → UniProt mapping (with cache)
# ══════════════════════════════════════════════════════════════
import os

if os.path.exists(UNIPROT_CACHE):
    print(f"Loading UniProt cache from {UNIPROT_CACHE}")
    nm_to_uniprot = (
        pd.read_csv(UNIPROT_CACHE, index_col=0)["uniprot_id"]
        .to_dict()
    )
else:
    def map_nm_to_uniprot_safe(nm_ids, batch_size=500, max_retries=3):
        mg = mygene.MyGeneInfo()
        nm_to_uniprot = {}

        for i in tqdm(range(0, len(nm_ids), batch_size), desc="MyGene mapping"):
            batch = nm_ids[i:i + batch_size]

            for attempt in range(max_retries):
                try:
                    results = mg.querymany(
                        batch,
                        scopes="refseq",
                        fields="uniprot",
                        species="human",
                        as_dataframe=False
                    )
                    for res in results:
                        refseq    = res.get("query")
                        uniprot   = res.get("uniprot", {})
                        uniprot_id = None
                        if isinstance(uniprot, dict):
                            ids = uniprot.get("Swiss-Prot") or uniprot.get("TrEMBL")
                            uniprot_id = ids[0] if isinstance(ids, list) else ids
                        elif isinstance(uniprot, list):
                            uniprot_id = uniprot[0] if uniprot else None
                        else:
                            uniprot_id = uniprot
                        nm_to_uniprot[refseq] = uniprot_id
                    break  # success

                except Exception as e:
                    print(f"  [batch {i} attempt {attempt+1}] {e}")
                    if attempt < max_retries - 1:
                        time.sleep(10 * (attempt + 1))
                    else:
                        print(f"  [SKIP] batch {i} failed after {max_retries} attempts")

        return nm_to_uniprot

    nm_ids = data["ClinVarName_refseq_ids"].dropna().unique().tolist()
    print(f"Mapping {len(nm_ids)} RefSeq IDs via MyGene...")
    nm_to_uniprot = map_nm_to_uniprot_safe(nm_ids)

    # save cache
    pd.Series(nm_to_uniprot).to_csv(UNIPROT_CACHE, header=["uniprot_id"])
    print(f"Cache saved to {UNIPROT_CACHE}")

data["uniprot_id"] = data["ClinVarName_refseq_ids"].map(nm_to_uniprot)
print(f"UniProt mapped: {data['uniprot_id'].notna().sum()} / {len(data)}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Build protein_variant string (AAREF + AAPOS + AAALT)
# ══════════════════════════════════════════════════════════════
data["protein_variant"] = (
    data["ClinVarName_AAREF"].astype(str) +
    data["ClinVarName_AAPOS"].astype("Int64").astype(str) +
    data["ClinVarName_AAALT"].astype(str)
)

needed_uniprots = set(data["uniprot_id"].dropna())
print(f"Need scores for {len(needed_uniprots)} UniProt IDs")

# ══════════════════════════════════════════════════════════════
# Read VESM++ parquet in chunks
#    - collect missense merge table
#    - collect per-protein variants for stop_gain
# ══════════════════════════════════════════════════════════════
pf   = pq.ParquetFile(VESM_PARQUET)
n_rg = pf.metadata.num_row_groups
print(f"VESM++ parquet has {n_rg} row groups")

vesm_parts            = []
protein_variants_vesm = {}   # {uniprot_id: df with AAPOS_int, VESM++}

for i in tqdm(range(n_rg), desc="Reading VESM++"):
    chunk = pf.read_row_group(i).to_pandas()
    chunk = chunk[chunk["uniprot_id"].isin(needed_uniprots)]
    if chunk.empty:
        continue

    # parse AAPOS from protein_variant (e.g. "V207L" → 207)
    chunk["AAPOS_int"] = pd.to_numeric(
        chunk["protein_variant"].str[1:-1], errors="coerce"
    )

    # missense merge table
    vesm_parts.append(
        chunk[["uniprot_id", "protein_variant", "VESM++"]].copy()
    )

    # stop_gain table: only need AAPOS and score
    stop_df = (
        chunk[["uniprot_id", "AAPOS_int", "VESM++"]]
        .dropna(subset=["AAPOS_int"])
        .copy()
    )
    stop_df["AAPOS_int"] = stop_df["AAPOS_int"].astype(int)

    for uni, subdf in stop_df.groupby("uniprot_id"):
        if uni not in protein_variants_vesm:
            protein_variants_vesm[uni] = subdf.copy()
        else:
            protein_variants_vesm[uni] = pd.concat(
                [protein_variants_vesm[uni], subdf], ignore_index=True
            )

    del chunk, stop_df
    gc.collect()

print(f"Collected stop_gain data for {len(protein_variants_vesm)} proteins")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Missense
# ══════════════════════════════════════════════════════════════

# 先建立data的lookup: (uniprot_id, protein_variant) → row indices
print("Building data lookup index...")
lookup = {}
for idx, row in data.iterrows():
    uni = row["uniprot_id"]
    pv  = row["protein_variant"]
    if pd.isna(uni) or pd.isna(pv) or str(pv) in ("nan", "nannannan"):
        continue
    key = (uni, pv)
    if key not in lookup:
        lookup[key] = []
    lookup[key].append(idx)

print(f"Lookup keys: {len(lookup)}")

# 初始化vesm_score欄位
data["vesm_score"] = np.nan

# 分chunk讀，match就直接寫入data
pf   = pq.ParquetFile(VESM_PARQUET)
n_rg = pf.metadata.num_row_groups
protein_variants_vesm = {}

for i in tqdm(range(n_rg), desc="Reading & merging VESM++"):
    chunk = pf.read_row_group(i).to_pandas()
    chunk = chunk[chunk["uniprot_id"].isin(needed_uniprots)]
    if chunk.empty:
        continue

    chunk["AAPOS_int"] = pd.to_numeric(
        chunk["protein_variant"].str[1:-1], errors="coerce"
    )

    # missense: 直接寫入data
    for _, vrow in chunk.iterrows():
        key = (vrow["uniprot_id"], vrow["protein_variant"])
        if key in lookup:
            for idx in lookup[key]:
                if pd.isna(data.at[idx, "vesm_score"]):
                    data.at[idx, "vesm_score"] = vrow["VESM++"]

    # stop_gain accumulate
    stop_df = (
        chunk[["uniprot_id", "AAPOS_int", "VESM++"]]
        .dropna(subset=["AAPOS_int"])
        .copy()
    )
    stop_df["AAPOS_int"] = stop_df["AAPOS_int"].astype(int)

    for uni, subdf in stop_df.groupby("uniprot_id"):
        if uni not in protein_variants_vesm:
            protein_variants_vesm[uni] = subdf.copy()
        else:
            protein_variants_vesm[uni] = pd.concat(
                [protein_variants_vesm[uni], subdf], ignore_index=True
            )

    del chunk, stop_df
    gc.collect()

n_missense = data["vesm_score"].notna().sum()
print(f"After missense merge: {n_missense} / {len(data)} have vesm_score")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Stop gain: take min (most damaging) score at or after stop pos
#    VESM++: more negative = more pathogenic → use min
# ══════════════════════════════════════════════════════════════
stop_mask    = data["ClinVarName_AAALT"].astype(str) == "*"
stop_indices = data.index[stop_mask]
print(f"Processing {len(stop_indices)} stop_gain variants...")

counter = 0
for idx in tqdm(stop_indices, desc="Stop gain VESM++"):
    row      = data.loc[idx]
    uni      = row["uniprot_id"]
    stop_pos = row["ClinVarName_AAPOS"]

    if pd.isna(uni) or pd.isna(stop_pos):
        continue

    stop_pos = int(float(stop_pos))

    if uni not in protein_variants_vesm:
        continue

    pv    = protein_variants_vesm[uni]
    after = pv[pv["AAPOS_int"] >= stop_pos]

    if after.empty:
        continue

    # most damaging = most negative
    data.at[idx, "vesm_score"] = after["VESM++"].min()
    counter += 1

print(f"Stop gain updated: {counter}")
print(f"Total vesm_score non-null: {data['vesm_score'].notna().sum()} / {len(data)}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Sanity check: direction of AUROC
#    Expect AUROC < 0.5 (more negative = more pathogenic = needs flip)
# ══════════════════════════════════════════════════════════════
missense_mask = (data["group: missense"] == 1) if "group: missense" in data.columns else pd.Series(True, index=data.index)
sub = data[
    missense_mask &
    data["vesm_score"].notna() &
    data["ClinVar_label"].notna()
]
if len(sub) > 0 and sub["ClinVar_label"].nunique() == 2:
    auc = roc_auc_score(sub["ClinVar_label"], -sub["vesm_score"])
    print(f"\nMissense AUROC (raw): {auc:.4f}")
    print(f"→ Score direction: {'needs flip (more negative = pathogenic, correct)' if auc < 0.5 else 'WARNING: unexpected direction'}")
else:
    print("Could not compute sanity check AUROC")

# ══════════════════════════════════════════════════════════════
# 診斷：分析 unmapped 原因（只看指定 coding 類群）
# ══════════════════════════════════════════════════════════════
TARGET_GROUPS = [
    "group: start loss",
    "group: missense",
    "group: missense + 3'UTR",
    "group: missense + intron (non-splice)",
    "group: stop gain",
]

available_groups = [g for g in TARGET_GROUPS if g in data.columns]
coding_mask = data[available_groups].eq(1).any(axis=1)

no_score = data[data["vesm_score"].isna() & coding_mask].copy()
print(f"\n=== Unmapped breakdown ({len(no_score)} rows, coding groups only) ===")
print(f"    Groups checked: {available_groups}")

# 原因 1：RefSeq → UniProt 就失敗了
no_uniprot = no_score[no_score["uniprot_id"].isna()]
print(f"\n[1] RefSeq → UniProt 失敗 (uniprot_id is NaN): {len(no_uniprot)}")

# 原因 2：有 UniProt 但 vesm_score 仍是 missing
has_uniprot = no_score[no_score["uniprot_id"].notna()]
print(f"[2] 有 UniProt 但 vesm_score missing: {len(has_uniprot)}")
print(f"    例如: {has_uniprot['uniprot_id'].value_counts().head(5).to_dict()}")

# ══════════════════════════════════════════════════════════════
# # Cleanup and save
# # ══════════════════════════════════════════════════════════════
# data = data.drop(columns=["uniprot_id", "protein_variant"], errors="ignore")

# data.to_csv(VARIANTS_CSV, index=False)
# print(f"\nSaved to {VARIANTS_CSV}")